In [ ]:
# -----------------------------------------
# Install packages
# -----------------------------------------
!pip install datasets efficientnet-pytorch torchvision scikit-learn seaborn grad-cam

# -----------------------------------------
# Imports
# -----------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from datasets import load_dataset
import os
import seaborn as sns
from sklearn.metrics import confusion_matrix
import random

# -----------------------------------------
# Load FaceForensics Dataset
# -----------------------------------------
from datasets import load_dataset

dataset = load_dataset("maxin-cn/FaceForensics", split="train")
print("Dataset loaded. Example:", dataset[0])

# -----------------------------------------
# Custom Dataset Class
# -----------------------------------------
class FaceForensicsDataset(Dataset):
    def __init__(self, dataset, seq_len=5, transform=None):
        self.dataset = dataset
        self.seq_len = seq_len
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        img = sample['image']
        label = sample['label']

        # Map labels: 0 = real, anything else = fake
        label = 0 if label == 0 else 1

        if self.transform:
            img = self.transform(img)

        frames = [img for _ in range(self.seq_len)]
        frames_tensor = torch.stack(frames)

        return frames_tensor, label
# -----------------------------------------
# Transforms and DataLoaders
# -----------------------------------------
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# Train/Val split manually
train_dataset = FaceForensicsDataset(dataset.select(range(0, 4000)), transform=transform)
val_dataset = FaceForensicsDataset(dataset.select(range(4000, 5000)), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

# -----------------------------------------
# Define Models
# -----------------------------------------
from efficientnet_pytorch import EfficientNet

class SpatialStreamModel(nn.Module):
    def __init__(self):
        super(SpatialStreamModel, self).__init__()
        self.backbone = EfficientNet.from_pretrained('efficientnet-b0')
        num_features = self.backbone._fc.in_features
        self.backbone._fc = nn.Linear(num_features, 2)

    def forward(self, x):
        return self.backbone(x)

class TemporalStreamModel(nn.Module):
    def __init__(self, feature_dim=64, lstm_hidden=32):
        super(TemporalStreamModel, self).__init__()
        self.cnn_features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(16, feature_dim, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.lstm = nn.LSTM(input_size=feature_dim, hidden_size=lstm_hidden, batch_first=True)
        self.fc = nn.Linear(lstm_hidden, 2)

    def forward(self, x):
        batch_size, seq_len, C, H, W = x.shape
        x_flat = x.view(batch_size * seq_len, C, H, W)
        frame_feats = self.cnn_features(x_flat)
        frame_feats = frame_feats.view(batch_size, seq_len, -1)
        lstm_out, _ = self.lstm(frame_feats)
        final_out = lstm_out[:, -1, :]
        logits = self.fc(final_out)
        return logits

class FusionModule(nn.Module):
    def __init__(self):
        super(FusionModule, self).__init__()

    def forward(self, spatial_logits, temporal_logits):
        return 0.5 * (spatial_logits + temporal_logits)

# Instantiate models
spatial_model = SpatialStreamModel()
temporal_model = TemporalStreamModel()
fusion_module = FusionModule()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
spatial_model.to(device)
temporal_model.to(device)
fusion_module.to(device)

# -----------------------------------------
# Training Setup
# -----------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(spatial_model.parameters()) + list(temporal_model.parameters()), lr=0.001)

def compute_metrics(preds, labels):
    preds = preds.argmax(dim=1)
    acc = (preds == labels).float().mean().item()
    tp = ((preds == 1) & (labels == 1)).sum().item()
    fp = ((preds == 1) & (labels == 0)).sum().item()
    fn = ((preds == 0) & (labels == 1)).sum().item()
    if tp + fp + fn == 0:
        f1 = 0
    else:
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return acc, f1
# -----------------------------------------
# Training Loop
# -----------------------------------------
num_epochs = 5
train_losses, val_losses = [], []
val_accuracies, val_f1s = [], []

for epoch in range(num_epochs):
    spatial_model.train()
    temporal_model.train()
    train_loss = 0.0

    for frames, labels in train_loader:
        frames, labels = frames.to(device), labels.to(device)
        optimizer.zero_grad()
        spatial_inputs = frames[:, 0, :, :, :]  # First frame only
        temporal_inputs = frames
        spatial_logits = spatial_model(spatial_inputs)
        temporal_logits = temporal_model(temporal_inputs)
        fused_logits = fusion_module(spatial_logits, temporal_logits)
        loss = criterion(fused_logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_losses.append(train_loss / len(train_loader))

    spatial_model.eval()
    temporal_model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for frames, labels in val_loader:
            frames, labels = frames.to(device), labels.to(device)
            spatial_inputs = frames[:, 0, :, :, :]
            temporal_inputs = frames
            spatial_logits = spatial_model(spatial_inputs)
            temporal_logits = temporal_model(temporal_inputs)
            fused_logits = fusion_module(spatial_logits, temporal_logits)
            loss = criterion(fused_logits, labels)
            val_loss += loss.item()
            all_preds.append(fused_logits)
            all_labels.append(labels)

    val_losses.append(val_loss / len(val_loader))
    preds_cat = torch.cat(all_preds, dim=0)
    labels_cat = torch.cat(all_labels, dim=0)
    val_acc, val_f1 = compute_metrics(preds_cat, labels_cat)
    val_accuracies.append(val_acc)
    val_f1s.append(val_f1)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

# -----------------------------------------
# Plotting Metrics
# -----------------------------------------
plt.figure(figsize=(8,6))
plt.plot(range(1, num_epochs+1), train_losses, label='Train Loss')
plt.plot(range(1, num_epochs+1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss over Epochs')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8,6))
plt.plot(range(1, num_epochs+1), val_accuracies, label='Validation Accuracy')
plt.plot(range(1, num_epochs+1), val_f1s, label='Validation F1-Score')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Validation Accuracy and F1-Score over Epochs')
plt.legend()
plt.grid(True)
plt.show()

# -----------------------------------------
# Confusion Matrix
# -----------------------------------------
predictions = preds_cat.argmax(dim=1).cpu()
true_labels = labels_cat.cpu()
cm = confusion_matrix(true_labels, predictions)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Validation Confusion Matrix")
plt.show()


# Save models after training

# Create a folder to save if not exists
os.makedirs('saved_models', exist_ok=True)

# Save Spatial Stream Model
torch.save(spatial_model.state_dict(), 'saved_models/spatial_stream_model.pth')

# Save Temporal Stream Model
torch.save(temporal_model.state_dict(), 'saved_models/temporal_stream_model.pth')

# Save Fusion Module (optional)
torch.save(fusion_module.state_dict(), 'saved_models/fusion_module.pth')


from google.colab import files

# Download spatial model
files.download('saved_models/spatial_stream_model.pth')

# Download temporal model
files.download('saved_models/temporal_stream_model.pth')

# Download fusion module (optional)
files.download('saved_models/fusion_module.pth')



# -----------------------------------------
# Upload Video and Predict Real/Fake
# -----------------------------------------
from google.colab import files

uploaded = files.upload()
video_path = next(iter(uploaded))

def extract_frames(video_path, frame_skip=5):
    cap = cv2.VideoCapture(video_path)
    frames = []
    count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if count % frame_skip == 0:
            frame = cv2.resize(frame, (64, 64))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = frame / 255.0
            frames.append(frame)
        count += 1
    cap.release()
    return np.array(frames)

frames = extract_frames(video_path)
frames_tensor = torch.tensor(frames).permute(0,3,1,2).unsqueeze(0).float()

spatial_model.eval()
temporal_model.eval()

with torch.no_grad():
    batch_size, seq_len, C, H, W = frames_tensor.shape
    spatial_preds = []
    for i in range(seq_len):
        frame = frames_tensor[:, i, :, :, :]
        spatial_logits = spatial_model(frame.to(device))
        spatial_score = torch.softmax(spatial_logits, dim=1)
        spatial_preds.append(spatial_score)
    spatial_preds = torch.cat(spatial_preds, dim=0)
    spatial_avg = torch.mean(spatial_preds, dim=0, keepdim=True)

    temporal_logits = temporal_model(frames_tensor.to(device))
    temporal_score = torch.softmax(temporal_logits, dim=1)

    fused_score = fusion_module(spatial_avg, temporal_score)
    pred_class = torch.argmax(fused_score, dim=1).item()

    classes = ['Real', 'Fake']
    print(f"Prediction: {classes[pred_class]}")
    print(f"Fused Probabilities -> Real: {fused_score[0][0]:.4f}, Fake: {fused_score[0][1]:.4f}")